### Checks

Merge `Urban Ward_Panel` and `Urban Ward_IFS` sheets. Add 2 columns, in_IFS and in_Panel.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- "FAIL - No shape(s) given", 
- "PASS - single Town/Village shape given (expected)", 
- "FAIL - only town shape given (but Ward shapes promised)", 
- "PASS - Ward shapes given", 
- "FAIL - Ward shapes given but sampled ward missing", 
- "PASS - subdistrict given only (expected)"

3. Also check population/summed up population matches the value in our reference.
4. Subdistrict codes for 3 states should show up for those states



#### OLD

- Sheet `Urban Ward_IFS` sheet: match the `TownVillage`-`UrbanWardVillage` columns (where it's not 0) to what's in the given dataset for each state - do these sampled wards exist in the shared shapes? Add column `Delivered by MapSolve`
    - If not, was it flagged as available? If yes flag as "Promised but not delivered ward shapes"
    - If yes, flag as delivered


`UrbanWardVillage`:
- If it's 0, it's a rural village, and there should just be NaN for Ward_C but one shape for the TV_Code (same as town)
- If it's non-zero, it's a sampled ward. If MapSolve has said available we should get wards. If they've said unavailable, the `Ward_C` should be null but the TV_Code should be there.


Only expect subdistrict boundaries for our sampled subdistricts - only check for subdistrict code existence:
- meghalaya
- tripura
- uttarakhand

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [1]:
# general
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm

# for plotting and coloring
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import math
import matplotlib.cm

gpd.options.io_engine = "pyogrio"

In [2]:
from gridsample.utils import create_ids, create_gmap_links, save_shapefiles
from gridsample.mapping import create_interactive_map

In [14]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = DATA_DIR / "02. Intermediate Outputs" / "00. Merge and Quality Checks" / "v2"
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [15]:
def generate_colormap(N):
    arr = np.arange(N)/N
    N_up = int(math.ceil(N/7)*7)
    arr.resize(N_up)
    arr = arr.reshape(7,N_up//7).T.reshape(-1)
    ret = matplotlib.cm.hsv(arr)
    n = ret[:,3].size
    a = n//2
    b = n-a
    
    # Create arrays of exactly matching sizes
    for i in range(3):
        ret[0:a,i] *= np.linspace(0.2, 1, a)
    ret[a:,3] *= np.linspace(1, 0.3, b)
    
    return ret[:N]  # Return only the requested number of colors

## 0. Load request excel

In [16]:
excel_file = RAW_DATA_DIR / "00. Boundary Requests" / "SamplingOutput_Summary_IFS & Panel.xlsx"
ifs_df = pd.read_excel(excel_file, sheet_name="Urban Ward_IFS")
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward_Panel")

In [17]:
ifs_df["Selected in IFS Sample"] = "Yes"

In [18]:
merged_df = panel_df.merge(
    ifs_df,
    on=list(ifs_df.columns.intersection(panel_df.columns)),
    how="outer",
    indicator="Source Sheet",
)
merged_df["Source Sheet"] = merged_df["Source Sheet"].replace(
    {
        "left_only": "Panel",
        "right_only": "IFS",
        "both": "Both Panel and IFS",
    }
)
merged_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Included in Panel,Selected in IFS Sample,Ward Boundary Available with MapSolve,Source Sheet
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No,Panel
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No,Panel
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No,Panel
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800165,7,Jalandhar Cantt. (CB) WARD NO.-0007,Urban,6349,"11,45,692","2,77,43,338",037-00212-800165-0007,No,No,Yes,Panel
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,1,Jalandhar (M Corp.) WARD NO.-0001,Urban,27750,"11,45,692","2,77,43,338",037-00212-800166-0001,No,No,Yes,Panel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,90,TELANGANA,536,Hyderabad,4509,Bandlaguda,802918,22,GHMC (M Corp.) (Part) WARD NO.-0022,Urban,47220,"3,30,193",NaN,536-04509-802918-0022,NaN,Yes,Yes,IFS
1015,90,TELANGANA,537,Rangareddy,4516,Serilingampally,802918,113,GHMC (M Corp.) (Part) WARD NO.-0113,Urban,78284,"3,09,320",NaN,537-04516-802918-0113,NaN,Yes,Yes,IFS
1016,90,TELANGANA,537,Rangareddy,4517,Balanagar,802918,120,GHMC (M Corp.) (Part) WARD NO.-0120,Urban,41373,"5,67,996",NaN,537-04517-802918-0120,NaN,Yes,Yes,IFS
1017,90,TELANGANA,537,Rangareddy,4517,Balanagar,802918,123,GHMC (M Corp.) (Part) WARD NO.-0123,Urban,67730,"5,67,996",NaN,537-04517-802918-0123,NaN,Yes,Yes,IFS


In [19]:
# rename columns for clarity
merged_df = merged_df.rename(
    columns={
        "Selected in IFS Sample": "Sampled for IFS",
        "Included in Panel": "Sampled for Panel",
    },
)

In [20]:
# check
print("Left:", merged_df.loc[merged_df["Source Sheet"] == "IFS", "Sampled for IFS"].value_counts())
print("Right or Both", merged_df.loc[merged_df["Source Sheet"] != "IFS", "Sampled for IFS"].value_counts())

Left: Sampled for IFS
Yes    307
Name: count, dtype: int64
Right or Both Sampled for IFS
No     666
Yes     46
Name: count, dtype: int64


In [21]:
# fill NaN values in "Included in Panel" column to No
merged_df["Sampled for Panel"].fillna("No", inplace=True)

# sort and reset index
merged_df = merged_df.sort_values(by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]).reset_index(drop=True)

In [22]:
merged_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Sampled for Panel,Sampled for IFS,Ward Boundary Available with MapSolve,Source Sheet
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,1,Shimla (M Corp.) WARD NO.-0001,Urban,4113,"1,69,578","68,64,602",033-00182-800137-0001,No,Yes,No,IFS
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No,Panel
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,19,Shimla (M Corp.) WARD NO.-0019,Urban,9627,"1,69,578","68,64,602",033-00182-800137-0019,No,Yes,No,IFS
3,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No,Panel
4,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No,Panel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574173,1,Peerzadguda (CT) WARD NO.-0001,Urban,32586,"1,88,380",NaN,537-04523-574173-0001,Yes,No,No,Panel
1015,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,4,Mahbubnagar (M) WARD NO.-0004,Urban,12175,"2,49,091",NaN,538-04568-802922-0004,Yes,Yes,No,Both Panel and IFS
1016,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579683,0,V. Venkatayapalem,Rural,3825,"3,13,504",NaN,541-04757-579683-0000,No,Yes,No,IFS
1017,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579685,1,Khanapuram Haveli (CT) WARD NO.-0001,Urban,53442,"3,13,504",NaN,541-04757-579685-0001,Yes,No,No,Panel


In [23]:
merged_df.to_csv(CLEANED_DATA_DIR / "Merged Panel and IFS Wards.csv", index=False)

In [24]:
# get unique across district and subdistrict both
len(merged_df[["State", "District", "Subdistrict"]].drop_duplicates())

275

In [25]:
subdistrict_only_state_names = ["MEGHALAYA", "TRIPURA", "UTTARAKHAND"]
merged_df.loc[merged_df["State_Name"].isin(subdistrict_only_state_names), ["State", "State_Name"]].value_counts()

State  State_Name 
5      UTTARAKHAND    44
16     TRIPURA        32
17     MEGHALAYA       7
Name: count, dtype: int64

In [26]:
subdistrict_only_state_codes = [17, 16, 5]

In [27]:
merged_df.columns

Index(['State', 'State_Name', 'District', 'District_Name', 'Subdistrict',
       'Subd_Name', 'TownVillage', 'UrbanWardVillage', 'WardVillage_Name',
       'TRU', 'WardVillage_Pop', 'Subd_Pop', 'State_Pop', 'WardVillageID',
       'Sampled for Panel', 'Sampled for IFS',
       'Ward Boundary Available with MapSolve', 'Source Sheet'],
      dtype='object')

#### Check for duplicated wards in our own sample

In [28]:
duplicated_sampled_wards = merged_df[
    merged_df[["TownVillage", "UrbanWardVillage"]].duplicated(keep=False)
].sort_values(["TownVillage", "UrbanWardVillage"])
duplicated_sampled_wards

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Sampled for Panel,Sampled for IFS,Ward Boundary Available with MapSolve,Source Sheet
135,7,NCT OF DELHI,90,North West,431,Saraswati Vihar,800441,59,DMC (U) (Part) WARD NO.-0059,Urban,45964,"22,50,816","1,67,87,941",090-00431-800441-0059,No,No,Yes,Panel
201,7,NCT OF DELHI,96,West,450,Punjabi Bagh,800441,59,DMC (U) (Part) WARD NO.-0059,Urban,10377,"7,99,453","1,67,87,941",096-00450-800441-0059,No,Yes,Yes,Both Panel and IFS
187,7,NCT OF DELHI,96,West,448,Patel Nagar,800441,99,DMC (U) (Part) WARD NO.-0099,Urban,35330,"12,62,158","1,67,87,941",096-00448-800441-0099,No,No,Yes,Panel
202,7,NCT OF DELHI,96,West,450,Punjabi Bagh,800441,99,DMC (U) (Part) WARD NO.-0099,Urban,16525,"7,99,453","1,67,87,941",096-00450-800441-0099,No,No,Yes,Panel
210,7,NCT OF DELHI,97,South West,451,Najafgarh,800441,136,DMC (U) (Part) WARD NO.-0136,Urban,80856,"13,65,152","1,67,87,941",097-00451-800441-0136,No,Yes,Yes,IFS
219,7,NCT OF DELHI,97,South West,453,Vasant Vihar,800441,136,DMC (U) (Part) WARD NO.-0136,Urban,35640,"6,41,666","1,67,87,941",097-00453-800441-0136,No,No,Yes,Panel
237,7,NCT OF DELHI,98,South,455,Defence Colony,800441,208,DMC (U) (Part) WARD NO.-0208,Urban,19918,"6,37,775","1,67,87,941",098-00455-800441-0208,No,Yes,Yes,IFS
244,7,NCT OF DELHI,98,South,456,Kalkaji,800441,208,DMC (U) (Part) WARD NO.-0208,Urban,16197,"8,62,861","1,67,87,941",098-00456-800441-0208,No,No,Yes,Panel


In [29]:
duplicated_sampled_wards.to_csv(
    CLEANED_DATA_DIR / "Duplicated Sampled Wards.csv", index=False
)

## 1. Load all boundaries

In [30]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [31]:
# # drop any with the word "Sub-District" in the filename
# gpkg_files_VTW = [f for f in gpkg_files_all if "Sub-District" not in f.name]

In [32]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

## 2. Checks

### Any MapSolve rows with missing TV code?

In [34]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

,UID,PCA_ID,State_C,State_N,Dist_C,Dist_N,SubDist_C,SubDist_N,TV_C,TV_N,Ward_C,TOT_P,geometry
24,16,None,22.0,Chhattisgarh,409.0,Durg,3317.0,Durg,NaN,None,NaN,NaN,"MULTIPOLYGON (((9037170.000 2420040.000, 90371..."
25,16,None,22.0,Chhattisgarh,406.0,Bilaspur,3295.0,Bilaspur,NaN,None,NaN,NaN,"MULTIPOLYGON (((9166590.000 2550750.000, 91665..."
1460,77,None,23.0,Madhya Pradesh,451.0,Jabalpur,3631.0,Jabalpur,NaN,None,NaN,NaN,"POLYGON ((8900430.000 2632320.000, 8900490.000..."
1650,9504321,55658,5.0,Uttarakhand,67.0,Udham Singh Nagar,346.0,Kashipur,NaN,None,NaN,283136.0,"MULTIPOLYGON (((8796240.000 3417120.000, 87963..."
1651,9488658,45145,5.0,Uttarakhand,60.0,Dehradun,304.0,Dehradun,NaN,None,NaN,988007.0,"MULTIPOLYGON (((8678280.000 3566610.000, 86783..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22517,240,7-95-447,7.0,NCT of Delhi,95.0,Central,447.0,Karol Bagh,NaN,NaN,NaN,136599.0,"MULTIPOLYGON (((8591760.000 3331200.000, 85918..."
22518,219,7-98-454,7.0,NCT of Delhi,98.0,South,454.0,Hauz Khas (Saket),NaN,NaN,NaN,1219100.0,"MULTIPOLYGON (((8595720.000 3300750.000, 85957..."
22519,155,7-97-453,7.0,NCT of Delhi,97.0,South West,453.0,Vasant Vihar,NaN,NaN,NaN,641666.0,"MULTIPOLYGON (((8586120.000 3312960.000, 85861..."
22520,156,7-97-452,7.0,NCT of Delhi,97.0,South West,452.0,Delhi Cantonment,NaN,NaN,NaN,286140.0,"MULTIPOLYGON (((8589120.000 3321540.000, 85891..."


In [35]:
gdf_no_TV_code_filtered.to_csv(CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False)

### Request satisfaction check

In [38]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(90) # manually add 90 for Telangana / Andhra Pradesh discrepency
len(given_states_list)

22

In [44]:
merged_df.loc[merged_df["State"].isin(given_states_list), "State_Name"].unique()

array(['HIMACHAL PRADESH', 'PUNJAB', 'UTTARAKHAND', 'HARYANA',
       'NCT OF DELHI', 'RAJASTHAN', 'UTTAR PRADESH', 'BIHAR', 'TRIPURA',
       'MEGHALAYA', 'JHARKHAND', 'ODISHA', 'CHHATTISGARH',
       'MADHYA PRADESH', 'GUJARAT', 'MAHARASHTRA', 'ANDHRA PRADESH',
       'KARNATAKA', 'KERALA', 'TAMIL NADU', 'TELANGANA'], dtype=object)

In [45]:
merged_df["State Shared by MapSolve"] = False
merged_df.loc[merged_df["State"].isin(given_states_list), "State Shared by MapSolve"] = True

In [49]:
merged_df[~merged_df["State Shared by MapSolve"]]

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Sampled for Panel,Sampled for IFS,Ward Boundary Available with MapSolve,Source Sheet,State Shared by MapSolve,State Changed
436,18,ASSAM,301,Dhubri,2008,Dhubri (Pt),281007,0,Kachuarkhas Pt.III,Rural,1084,"4,25,809","3,12,05,576",301-02008-281007-0000,Yes,No,No,Panel,False,No
437,18,ASSAM,301,Dhubri,2008,Dhubri (Pt),281020,0,Baldiaralga N.C.,Rural,3297,"4,25,809","3,12,05,576",301-02008-281020-0000,No,Yes,No,IFS,False,No
438,18,ASSAM,301,Dhubri,2008,Dhubri (Pt),801549,11,Dhubri (MB) WARD NO.-0011,Urban,6233,"4,25,809","3,12,05,576",301-02008-801549-0011,Yes,No,Yes,Panel,False,No
439,18,ASSAM,301,Dhubri,2012,Mankachar,281650,0,Berabanga Pt.II,Rural,3317,"3,05,606","3,12,05,576",301-02012-281650-0000,No,Yes,No,IFS,False,No
440,18,ASSAM,301,Dhubri,2012,Mankachar,281676,0,Kanaimara Pt.II,Rural,2023,"3,05,606","3,12,05,576",301-02012-281676-0000,Yes,No,No,Panel,False,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,19,WEST BENGAL,345,Purba Medinipur,2473,Sahid Matangini,344650,1,Shantipur (CT) WARD NO.-0001,Urban,9746,"1,99,210","9,12,76,115",345-02473-344650-0001,No,Yes,No,IFS,False,No
538,19,WEST BENGAL,345,Purba Medinipur,2475,Mahisadal,344763,0,Gopalpur,Rural,8294,"2,06,277","9,12,76,115",345-02475-344763-0000,No,No,No,Panel,False,No
539,19,WEST BENGAL,345,Purba Medinipur,2475,Mahisadal,344806,0,Natshal,Rural,11331,"2,06,277","9,12,76,115",345-02475-344806-0000,No,No,No,Panel,False,No
540,19,WEST BENGAL,345,Purba Medinipur,2476,Moyna,344857,0,Paramanandapur,Rural,8026,"2,26,927","9,12,76,115",345-02476-344857-0000,No,No,No,Panel,False,No


In [50]:
merged_df["State Changed"] = "No"
merged_df.loc[merged_df["State_Name"] == "TELANGANA", "State Changed"] = "Previously Andhra Pradesh"

In [51]:
merged_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Sampled for Panel,Sampled for IFS,Ward Boundary Available with MapSolve,Source Sheet,State Shared by MapSolve,State Changed
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,1,Shimla (M Corp.) WARD NO.-0001,Urban,4113,"1,69,578","68,64,602",033-00182-800137-0001,No,Yes,No,IFS,True,No
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No,Panel,True,No
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,19,Shimla (M Corp.) WARD NO.-0019,Urban,9627,"1,69,578","68,64,602",033-00182-800137-0019,No,Yes,No,IFS,True,No
3,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No,Panel,True,No
4,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No,Panel,True,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574173,1,Peerzadguda (CT) WARD NO.-0001,Urban,32586,"1,88,380",NaN,537-04523-574173-0001,Yes,No,No,Panel,True,Previously Andhra Pradesh
1015,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,4,Mahbubnagar (M) WARD NO.-0004,Urban,12175,"2,49,091",NaN,538-04568-802922-0004,Yes,Yes,No,Both Panel and IFS,True,Previously Andhra Pradesh
1016,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579683,0,V. Venkatayapalem,Rural,3825,"3,13,504",NaN,541-04757-579683-0000,No,Yes,No,IFS,True,Previously Andhra Pradesh
1017,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579685,1,Khanapuram Haveli (CT) WARD NO.-0001,Urban,53442,"3,13,504",NaN,541-04757-579685-0001,Yes,No,No,Panel,True,Previously Andhra Pradesh


#### Check for wards

In [52]:
merged_df["PCA_ID"] = merged_df["TownVillage"].astype(str) + "-" + merged_df["UrbanWardVillage"].astype(str)
given_ward_ids = gdf["PCA_ID"].unique()

merged_df["Ward Boundary Given"] = False
merged_df.loc[merged_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [53]:
len(set(merged_df["PCA_ID"]).intersection(given_ward_ids))

412

#### Check for TownVillage codes

In [54]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

merged_df["TV Boundary Given"] = False
merged_df.loc[
    merged_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [55]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

merged_df["SubDistrict Boundary Given"] = False
merged_df.loc[
    merged_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

In [56]:
# - "FAIL - No shape(s) given", 
# - "PASS - Ward shapes given", 
# - "FAIL - only town shape given (but Ward shapes promised)", 
# - "PASS - single Town/Village shape given (expected)", 
# - "PASS - subdistrict given only (expected)"

# not implemented:
# - "FAIL - Ward shapes given but sampled ward missing", 

In [57]:
## baseline
merged_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
merged_df.loc[
    (merged_df["State_Name"] == "TRIPURA") & (merged_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

#### Add new columns to sample data: `PSU Type` and `Ward Count`

In [58]:
merged_df["Delivery State"].unique()

array(['BETTER - Ward boundary given but only TV or Subdistrict was expected',
       'GOOD - Town/Village boundary given as expected',
       'BAD - Town/Village boundary given but Ward boundaries expected',
       'GOOD - Ward boundary given as expected',
       'GOOD - Subdistrict boundary given as expected',
       'BAD - Subdistrict boundary given but Ward boundaries expected',
       'BAD - No boundary(s) given'], dtype=object)

In [59]:
psu_mapping = {
    "GOOD - Subdistrict boundary given as expected": "subdistrict",
    "GOOD - Ward boundary given as expected": "ward",
    "GOOD - Town/Village boundary given as expected": "town_village",
    "BAD - No boundary(s) given": "none",
    "BAD - Town/Village boundary given but Ward boundaries expected": "town_village",
    "BETTER - Ward boundary given but only TV or Subdistrict was expected": "ward",
}
# create a new column in merged_df called PSU Type
merged_df["PSU Type"] = merged_df["Delivery State"].map(psu_mapping)

In [60]:
merged_df["Ward Count"] = 0

merged_df.loc[merged_df["PSU Type"] == "ward", "Ward Count"] = 1
merged_df.loc[merged_df["PSU Type"] == "town_village", "Ward Count"] = (
    merged_df[merged_df["PSU Type"] == "town_village"].groupby("TownVillage")["UrbanWardVillage"].transform("nunique")
)
merged_df.loc[merged_df["PSU Type"] == "subdistrict", "Ward Count"] = (
    merged_df[merged_df["PSU Type"] == "subdistrict"].groupby("Subdistrict")["UrbanWardVillage"].transform("nunique")
)

In [62]:
merged_df["Delivery State"].value_counts()

Delivery State
GOOD - Ward boundary given as expected                                  403
GOOD - Town/Village boundary given as expected                          380
BAD - No boundary(s) given                                              116
GOOD - Subdistrict boundary given as expected                            95
BETTER - Ward boundary given but only TV or Subdistrict was expected     13
BAD - Town/Village boundary given but Ward boundaries expected            9
BAD - Subdistrict boundary given but Ward boundaries expected             3
Name: count, dtype: int64

#### Save

In [63]:
merged_df.to_csv(CLEANED_DATA_DIR / "Merged Wards with Quality Checks.csv", index=False)

In [64]:
merged_df[merged_df["State Shared by MapSolve"]].to_csv(
    CLEANED_DATA_DIR / "Merged Wards with Quality Checks - Shared States.csv",
    index=False,
)

#### Save per-state stats

In [73]:
# Get counts of delivery states by state
delivery_state_counts = (
    merged_df[merged_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    merged_df[["State", "State_Name"]]
    .drop_duplicates()
    .set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

State_Name,Delivery State,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,TRIPURA,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,0,0,0,0,0,0,2,2,0,...,0,0,0,1,1,0,0,0,1,1
1,BAD - Subdistrict boundary given but Ward boun...,0,0,2,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,BAD - Town/Village boundary given but Ward bou...,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,6,0,0
3,BETTER - Ward boundary given but only TV or Su...,5,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,4,0
4,GOOD - Subdistrict boundary given as expected,0,0,42,0,42,0,0,0,5,...,0,1,0,0,0,0,0,0,0,0
5,GOOD - Town/Village boundary given as expected,0,16,0,15,0,1,57,46,27,...,14,12,7,11,22,35,5,34,32,15
6,GOOD - Ward boundary given as expected,0,23,0,13,86,11,21,11,0,...,14,14,17,41,38,0,41,5,24,26


In [74]:
# add a total column as the second column
delivery_state_pivot.insert(1, "Total", delivery_state_pivot.iloc[:,1:].sum(axis=1))

In [75]:
delivery_state_pivot

State_Name,Delivery State,Total,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,10,0,0,0,0,0,0,2,2,...,0,0,0,1,1,0,0,0,1,1
1,BAD - Subdistrict boundary given but Ward boun...,3,0,0,2,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,BAD - Town/Village boundary given but Ward bou...,9,0,1,0,0,0,0,0,1,...,0,0,0,0,0,1,0,6,0,0
3,BETTER - Ward boundary given but only TV or Su...,13,5,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,4,0
4,GOOD - Subdistrict boundary given as expected,95,0,0,42,0,42,0,0,0,...,0,1,0,0,0,0,0,0,0,0
5,GOOD - Town/Village boundary given as expected,380,0,16,0,15,0,1,57,46,...,14,12,7,11,22,35,5,34,32,15
6,GOOD - Ward boundary given as expected,403,0,23,0,13,86,11,21,11,...,14,14,17,41,38,0,41,5,24,26


In [76]:
delivery_state_pivot.to_csv(CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False)

#### Old

In [ ]:
unique_requested_tv_codes = set(merged_df[merged_df["State Shared by MapSolve"]]["TownVillage"])
unique_received_tv_codes = set(gdf.loc[~gdf["TV_C"].isna(), "TV_C"].astype(int))
matched_tv_codes = unique_received_tv_codes.intersection(unique_requested_tv_codes)

In [ ]:
print("Number of requested TV codes:", len(unique_requested_tv_codes))
print("Number of received TV codes:", len(unique_received_tv_codes))

print("Number of matched TV codes:", len(matched_tv_codes))
print("Number of TV codes not received:", len(unique_requested_tv_codes.difference(matched_tv_codes)))

In [ ]:
# show the received breakdown by state
received_pivot_table = (
    merged_df[merged_df["State Shared by MapSolve"]]
    .groupby(
        [
            "State",
            "State_Name",
            "District",
            "District_Name",
            "Subdistrict",
            "Subd_Name",
        ]
    )["Delivery State"]
    .value_counts()
    .unstack(fill_value=0)
).reset_index()

# received_pivot_table.rename(
#     columns={True: "TV_Codes Received", False: "TV_Codes Not Received"}, inplace=True
# )

In [ ]:
received_pivot_table

In [ ]:
pivot_table_not_received = received_pivot_table[
    received_pivot_table["BAD - No boundary(s) given"] > 0
]
pivot_table_not_received


In [ ]:
pivot_table_not_received.to_csv(
    CLEANED_DATA_DIR / "Requested TV Codes Not Received Subdistrict Breakdown.csv", index=False
)

In [ ]:
requested_tv_codes_not_received = merged_df[
    (merged_df["State Shared by MapSolve"])
    & (merged_df["Delivery State"] == "BAD - No boundary(s) given")
]
requested_tv_codes_not_received

In [ ]:
requested_tv_codes_not_received.to_csv(
    CLEANED_DATA_DIR / "Requested TV Codes Not Received.csv", index=False
)

##### Tripura - they gave us the subdistrict boundary for Dukli

In [ ]:
gdf[(gdf["State_N"] == "Tripura")]

In [ ]:
ax = gdf[(gdf["State_N"] == "Tripura")].plot()
gdf[gdf["SubDist_N"] == "Dukli"].plot(ax=ax, color="red")

### Others

In [ ]:
# boundaries_gdf = gpd.read_file(RAW_DATA_DIR / "01. MapSolve Boundaries/State_C_7/State_C_7_Sub-District.gpkg")
# boundaries_gdf = boundaries_gdf.to_crs(4326)

# boundaries_gdf.plot(
#     column="PCA_ID",
#     figsize=(6, 6),
#     cmap=ListedColormap(generate_colormap(len(boundaries_gdf))),
#     edgecolor="black",
# )
# # create_interactive_map(boundaries_gdf, point_id_col="PCA_ID")


In [ ]:
# boundaries_gdf.plot(
#     column="PCA_ID",
#     figsize=(6, 6),
#     cmap=ListedColormap(generate_colormap(len(boundaries_gdf))),
#     edgecolor="black",
# )
# create_interactive_map(boundaries_gdf, point_id_col="PCA_ID")